# pandas Interview Refresher

Practice selection, missing-data repair, combination, groupby, windows, reshaping, joins, strings, and dates with traceable outputs.

- **Study time:** 40-50 minutes
- **Prerequisites:** NumPy arrays and basic SQL-style grouping
- **Mode:** `quick`
- **Data policy:** no external files or downloads; a deterministic event table is created in memory
- **Provenance:** rebuilt from the curated pandas interview recap notebook

Output convention: every retained textual result begins with a label that identifies the operation that produced it.
Annotation convention: comments explain intent, shape changes, invariants, subtle API behavior, or configuration side effects; obvious Python syntax is left uncommented.


In [1]:
import numpy as np
import pandas as pd

events = pd.DataFrame(
    {
        "event_id": np.arange(1, 13),
        "user_id": [101, 101, 102, 101, 103, 102, 103, 103, 101, 102, 103, 102],
        "timestamp": pd.to_datetime(
            [
                "2026-01-01 09:00",
                "2026-01-01 11:00",
                "2026-01-01 09:30",
                "2026-01-02 08:00",
                "2026-01-02 10:15",
                "2026-01-03 12:00",
                "2026-01-03 12:30",
                "2026-01-04 14:00",
                "2026-01-05 09:00",
                "2026-01-05 10:00",
                "2026-01-05 11:00",
                "2026-01-06 16:00",
            ]
        ),
        "channel": [
            "web",
            "app",
            "web",
            "store",
            "app",
            "web",
            "store",
            "app",
            "web",
            "store",
            "web",
            "app",
        ],
        "revenue": [20, 35, 15, 60, 10, 45, 55, 25, 80, 30, 50, 70],
        "note": [f"order_id=ORD-{value:03d}" for value in range(1, 13)],
    }
)
print("Source | event table", events)
print("Source | shape and dtypes", (events.shape, events.dtypes.astype(str).to_dict()))

Source | event table     event_id  user_id           timestamp channel  revenue              note
0          1      101 2026-01-01 09:00:00     web       20  order_id=ORD-001
1          2      101 2026-01-01 11:00:00     app       35  order_id=ORD-002
2          3      102 2026-01-01 09:30:00     web       15  order_id=ORD-003
3          4      101 2026-01-02 08:00:00   store       60  order_id=ORD-004
4          5      103 2026-01-02 10:15:00     app       10  order_id=ORD-005
5          6      102 2026-01-03 12:00:00     web       45  order_id=ORD-006
6          7      103 2026-01-03 12:30:00   store       55  order_id=ORD-007
7          8      103 2026-01-04 14:00:00     app       25  order_id=ORD-008
8          9      101 2026-01-05 09:00:00     web       80  order_id=ORD-009
9         10      102 2026-01-05 10:00:00   store       30  order_id=ORD-010
10        11      103 2026-01-05 11:00:00     web       50  order_id=ORD-011
11        12      102 2026-01-06 16:00:00     app      

## 1. `loc`, `iloc`, and safe assignment


In [2]:
high_value = events.loc[events["revenue"] >= 50, ["event_id", "user_id", "revenue"]]
high_value_web = events.loc[
    (events["revenue"] >= 50)
    & (events["channel"] == "web"),  # Parenthesize each mask around & / |.
    ["event_id", "channel", "revenue"],
]
positional = events.iloc[:3, :4]
labeled = events.copy()  # Make ownership explicit before adding a column.
labeled.loc[labeled["revenue"] >= 50, "value_band"] = "high"
labeled.loc[labeled["revenue"] < 50, "value_band"] = "regular"

print("Selection | loc revenue >= 50", high_value)
print("Selection | parenthesized AND mask", high_value_web)
print("Selection | iloc first 3 rows and 4 columns", positional)
print("Assignment | value_band counts", labeled["value_band"].value_counts())

Selection | loc revenue >= 50     event_id  user_id  revenue
3          4      101       60
6          7      103       55
8          9      101       80
10        11      103       50
11        12      102       70
Selection | parenthesized AND mask     event_id channel  revenue
8          9     web       80
10        11     web       50
Selection | iloc first 3 rows and 4 columns    event_id  user_id           timestamp channel
0         1      101 2026-01-01 09:00:00     web
1         2      101 2026-01-01 11:00:00     app
2         3      102 2026-01-01 09:30:00     web
Assignment | value_band counts value_band
regular    7
high       5
Name: count, dtype: int64


## 2. Missing values and dtype repair

Audit missingness before choosing a policy. Coerce dirty numeric text with `errors="coerce"`, impute features only from training data, and normally drop rather than impute a missing supervised target.


In [3]:
messy = events.copy()
messy.loc[2, "revenue"] = np.nan
messy.loc[5, "channel"] = None
messy["revenue_text"] = messy["revenue"].astype("string")
messy.loc[4, "revenue_text"] = "unknown"

numeric_revenue = pd.to_numeric(
    messy["revenue_text"], errors="coerce"
)  # Invalid text becomes NaN for audit/repair.
cleaned = messy.copy()
cleaned["revenue"] = numeric_revenue.fillna(
    numeric_revenue.median()
)  # In ML, learn this fill value on training rows only.
cleaned["channel"] = messy["channel"].fillna("unknown")
print("Missing data | counts before repair", messy.isna().sum())
print("Dtype repair | coerced invalid numeric values", numeric_revenue.head(6))
print("Missing data | counts after selected repairs", cleaned.isna().sum())

Missing data | counts before repair event_id        0
user_id         0
timestamp       0
channel         1
revenue         1
note            0
revenue_text    1
dtype: int64
Dtype repair | coerced invalid numeric values 0    20.0
1    35.0
2    <NA>
3    60.0
4    <NA>
5    45.0
Name: revenue_text, dtype: Float64
Missing data | counts after selected repairs event_id        0
user_id         0
timestamp       0
channel         0
revenue         0
note            0
revenue_text    1
dtype: int64


## 3. Concatenating compatible tables

`concat` appends already-compatible tables along an axis; it does not match rows by key. Here the original table is split and rebuilt by rows. `ignore_index=True` replaces the two inherited index fragments with one continuous index.


In [4]:
first_rows = events.iloc[:5]
remaining_rows = events.iloc[5:]
recombined = pd.concat([first_rows, remaining_rows], ignore_index=True)

print("Concat | input shapes", (first_rows.shape, remaining_rows.shape))
print("Concat | row-wise result shape", recombined.shape)
print("Concat | rows around the join point", recombined.iloc[3:7])

Concat | input shapes ((5, 6), (7, 6))
Concat | row-wise result shape (12, 6)
Concat | rows around the join point    event_id  user_id           timestamp channel  revenue              note
3         4      101 2026-01-02 08:00:00   store       60  order_id=ORD-004
4         5      103 2026-01-02 10:15:00     app       10  order_id=ORD-005
5         6      102 2026-01-03 12:00:00     web       45  order_id=ORD-006
6         7      103 2026-01-03 12:30:00   store       55  order_id=ORD-007


## 4. Sorting, deduplication, and top-k per group


In [5]:
latest_per_user = (
    events.sort_values(
        ["user_id", "timestamp", "event_id"]
    )  # event_id resolves timestamp ties deterministically.
    .drop_duplicates("user_id", keep="last")  # Sorted last row is the latest per user.
    .sort_values("user_id")
)
top_two = (
    events.sort_values(
        ["user_id", "revenue"], ascending=[True, False]
    )  # Put each group's winners first.
    .groupby("user_id", group_keys=False)
    .head(2)
)

print("Dedup | latest event per user", latest_per_user[["user_id", "event_id", "timestamp"]])
print("Ranking | top 2 revenue events per user", top_two[["user_id", "event_id", "revenue"]])

Dedup | latest event per user     user_id  event_id           timestamp
8       101         9 2026-01-05 09:00:00
11      102        12 2026-01-06 16:00:00
10      103        11 2026-01-05 11:00:00
Ranking | top 2 revenue events per user     user_id  event_id  revenue
8       101         9       80
3       101         4       60
11      102        12       70
5       102         6       45
6       103         7       55
10      103        11       50


## 5. `agg` reduces rows; `transform` preserves rows


In [6]:
user_summary = events.groupby("user_id").agg(  # Collapse to one row per user.
    event_count=("event_id", "size"),
    total_revenue=("revenue", "sum"),
    mean_revenue=("revenue", "mean"),
)
with_group_features = events.copy()
with_group_features["user_mean_revenue"] = events.groupby("user_id")["revenue"].transform(
    "mean"
)  # Broadcast one group statistic back to every event.
with_group_features["above_user_mean"] = (
    with_group_features["revenue"] > with_group_features["user_mean_revenue"]
)

print("Groupby agg | one row per user", user_summary)
print(
    "Groupby transform | row-aligned feature sample",
    with_group_features[
        ["event_id", "user_id", "revenue", "user_mean_revenue", "above_user_mean"]
    ].head(8),
)

Groupby agg | one row per user          event_count  total_revenue  mean_revenue
user_id                                          
101                4            195         48.75
102                4            160         40.00
103                4            140         35.00
Groupby transform | row-aligned feature sample    event_id  user_id  revenue  user_mean_revenue  above_user_mean
0         1      101       20              48.75            False
1         2      101       35              48.75            False
2         3      102       15              40.00            False
3         4      101       60              48.75             True
4         5      103       10              35.00            False
5         6      102       45              40.00             True
6         7      103       55              35.00             True
7         8      103       25              35.00            False


## 6. Time ordering, shift, gaps, and lagged rolling features


In [7]:
ordered = events.sort_values(
    ["user_id", "timestamp", "event_id"]
).copy()  # Temporal operations require explicit order.
ordered["previous_timestamp"] = ordered.groupby("user_id")["timestamp"].shift(
    1
)  # Never cross user boundaries.
ordered["gap_hours"] = (
    ordered["timestamp"] - ordered["previous_timestamp"]
).dt.total_seconds() / 3600
ordered["previous_revenue"] = ordered.groupby("user_id")["revenue"].shift(
    1
)  # Lag before rolling to exclude the current event.
ordered["prior_two_mean"] = (
    ordered.groupby("user_id")["previous_revenue"]
    .rolling(2, min_periods=1)  # Emit an early value when only one prior event exists.
    .mean()
    .reset_index(level=0, drop=True)  # Remove the group level so values align to ordered's index.
)
ordered["month"] = ordered["timestamp"].dt.to_period(
    "M"
)  # Calendar month period, not a formatted display string.

print(
    "Time features | previous event, gap, lagged rolling mean",
    ordered[["user_id", "timestamp", "revenue", "gap_hours", "prior_two_mean"]],
)

Time features | previous event, gap, lagged rolling mean     user_id           timestamp  revenue  gap_hours  prior_two_mean
0       101 2026-01-01 09:00:00       20        NaN             NaN
1       101 2026-01-01 11:00:00       35       2.00            20.0
3       101 2026-01-02 08:00:00       60      21.00            27.5
8       101 2026-01-05 09:00:00       80      73.00            47.5
2       102 2026-01-01 09:30:00       15        NaN             NaN
5       102 2026-01-03 12:00:00       45      50.50            15.0
9       102 2026-01-05 10:00:00       30      46.00            30.0
11      102 2026-01-06 16:00:00       70      30.00            37.5
4       103 2026-01-02 10:15:00       10        NaN             NaN
6       103 2026-01-03 12:30:00       55      26.25            10.0
7       103 2026-01-04 14:00:00       25      25.50            32.5
10      103 2026-01-05 11:00:00       50      21.00            40.0


## 7. Strings and categorical cleanup


In [8]:
string_features = events[["event_id", "note", "channel"]].copy()
string_features["order_id"] = string_features["note"].str.extract(
    r"(ORD-\d+)"
)  # The capture group becomes the new column.
string_features["channel"] = string_features["channel"].astype(
    "category"
)  # Store repeated labels as categorical levels.

print("Strings | extracted order identifiers", string_features.head(6))
print("Categorical | channel categories", string_features["channel"].cat.categories.tolist())

Strings | extracted order identifiers    event_id              note channel order_id
0         1  order_id=ORD-001     web  ORD-001
1         2  order_id=ORD-002     app  ORD-002
2         3  order_id=ORD-003     web  ORD-003
3         4  order_id=ORD-004   store  ORD-004
4         5  order_id=ORD-005     app  ORD-005
5         6  order_id=ORD-006     web  ORD-006
Categorical | channel categories ['app', 'store', 'web']


## 8. Pivot, pivot table, and melt


In [9]:
revenue_matrix = events.pivot_table(  # Aggregate duplicate user/channel pairs while widening.
    index="user_id",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
)
revenue_matrix.columns = [f"revenue_{column}" for column in revenue_matrix.columns]
wide = revenue_matrix.reset_index()
long = wide.melt(
    id_vars="user_id", var_name="metric", value_name="value"
)  # Return metric columns to tidy rows.

print("Reshape | revenue pivot table", wide)
print("Reshape | melted long form", long.head(9))

Reshape | revenue pivot table    user_id  revenue_app  revenue_store  revenue_web
0      101           35             60          100
1      102           70             30           60
2      103           35             55           50
Reshape | melted long form    user_id         metric  value
0      101    revenue_app     35
1      102    revenue_app     70
2      103    revenue_app     35
3      101  revenue_store     60
4      102  revenue_store     30
5      103  revenue_store     55
6      101    revenue_web    100
7      102    revenue_web     60
8      103    revenue_web     50


## 9. Validated joins and unmatched-key checks


In [10]:
users = pd.DataFrame(
    {
        "user_id": [101, 102, 103, 104],
        "segment": ["growth", "core", "growth", "new"],
    }
)
joined = events.merge(
    users,
    on="user_id",
    how="left",
    validate="many_to_one",  # Fail if the supposed lookup table would multiply event rows.
    indicator=True,  # Retain row-level match provenance for the join audit.
)
unmatched = joined.loc[joined["_merge"] != "both", ["event_id", "user_id", "_merge"]]

print("Join | events enriched with segment", joined.head(8))
print("Join audit | unmatched event keys", unmatched)

Join | events enriched with segment    event_id  user_id           timestamp channel  revenue              note  \
0         1      101 2026-01-01 09:00:00     web       20  order_id=ORD-001   
1         2      101 2026-01-01 11:00:00     app       35  order_id=ORD-002   
2         3      102 2026-01-01 09:30:00     web       15  order_id=ORD-003   
3         4      101 2026-01-02 08:00:00   store       60  order_id=ORD-004   
4         5      103 2026-01-02 10:15:00     app       10  order_id=ORD-005   
5         6      102 2026-01-03 12:00:00     web       45  order_id=ORD-006   
6         7      103 2026-01-03 12:30:00   store       55  order_id=ORD-007   
7         8      103 2026-01-04 14:00:00     app       25  order_id=ORD-008   

  segment _merge  
0  growth   both  
1  growth   both  
2    core   both  
3  growth   both  
4  growth   both  
5    core   both  
6  growth   both  
7  growth   both  
Join audit | unmatched event keys Empty DataFrame
Columns: [event_id, user_id, _m

## 10. Retrieval checks


In [11]:
assert top_two.groupby("user_id").size().eq(2).all()
assert joined["event_id"].is_unique
assert recombined.equals(events)
assert cleaned[["revenue", "channel"]].notna().all().all()
assert long.shape[0] == len(wide) * len(revenue_matrix.columns)
assert (
    ordered.groupby("user_id")["timestamp"]
    .apply(lambda values: values.is_monotonic_increasing)
    .all()
)

print("Drill checks | status", "all assertions passed")

Drill checks | status all assertions passed
